# Demonstração: Pipeline de Processamento de Textos para RAG
Este notebook tem como objetivo demonstrar o passo a passo de como um texto bruto (extraído de um PDF) é limpo e dividido em _chunks_ (segmentos semânticos) no nosso pipeline de dados.

## 1. Clonando o Repositório e Baixando a Amostra
Em vez de precisarmos fazer o upload manual dos PDFs para o Colab, vamos baixar o nosso próprio repositório diretamente do GitHub, que já contém uma pasta de `amostra` com os PDFs de exemplo prontos para uso.

In [ ]:
# Baixa os arquivos e a pasta de amostra do GitHub
!git clone https://github.com/abraaonazario/observatorio-ia.git

# Instala as bibliotecas de Processamento Natural de Linguagem (NLP) necessárias
!pip install spacy pandas nltk tqdm fpdf pdfplumber
!python -m spacy download pt_core_news_sm

import nltk
nltk.download('stopwords')
nltk.download('punkt')

## 2. A Classe Base do Pipeline (SemanticChunker)
Abaixo está a classe que utilizamos para limpar ruídos (como cabeçalhos de sites) e dividir as notícias em blocos com sentido (chunks), preservando o contexto através de *overlap*.

In [ ]:
import re
import spacy
import unicodedata
import string
from nltk.corpus import stopwords

class SemanticChunker:
    def __init__(self, chunk_size=350, overlap=100):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.nlp = spacy.load('pt_core_news_sm')

    def clean_text(self, text):
        if not text or not isinstance(text, str):
            return ''
        text = unicodedata.normalize('NFKC', text)
        # Filtros de ruidos web e LGPD
        text = re.sub(r'(?i)(Aceitar todos os cookies|Política de Privacidade|Este site usa cookies).*?(?=\n|\.)', ' ', text)
        text = re.sub(r'(?i)(Versão digital|Buscar Menu Geral|Esportes Entretenimento|Polícia Política|ELEIÇÕES \d{4}).*?(?=\n|\.)', ' ', text)
        text = re.sub(r'(?i)(Página\s+\d+\s+de\s+\d+|\d+\s*/\s*\d+|Impresso por:?\s*.*?\n|Gerado em:?\s*.*?\n)', ' ', text)
        text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text)
        text = re.sub(r'https?://[^\s]+', ' ', text)
        text = re.sub(r'\(cid:\d+\)', ' ', text)
        text = re.sub(r'-\s*\n\s*', '', text)
        text = re.sub(r'([a-zçãõáéíóú])\s*\n\s*([a-zçãõáéíóú])', r'\1\2', text)
        text = re.sub(r'[\t\r\v\f]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def split_into_semantic_chunks(self, text):
        cleaned = self.clean_text(text)
        if not cleaned:
            return []
        doc = self.nlp(cleaned)
        stop_words = set(stopwords.words('portuguese'))
        
        sentences = []
        for sent in doc.sents:
            s_text = sent.text.strip().lower()
            s_text = s_text.translate(str.maketrans('', '', string.punctuation))
            s_text = ' '.join([w for w in s_text.split() if w not in stop_words and not w.isnumeric()])
            if len(s_text) > 3:
                sentences.append(s_text)
        
        chunks = []
        current_chunk = []
        current_len = 0
        
        for sentence in sentences:
            sent_len = len(sentence)
            if sent_len >= self.chunk_size:
                if current_chunk:
                    chunks.append(' '.join(current_chunk))
                    current_chunk = []
                    current_len = 0
                chunks.append(sentence)
                continue
                
            if current_len + sent_len + 1 > self.chunk_size:
                chunks.append(' '.join(current_chunk))
                if len(current_chunk) >= 1 and len(current_chunk[-1]) <= self.chunk_size // 2:
                    current_chunk = [current_chunk[-1], sentence]
                    current_len = len(current_chunk[0]) + sent_len + 1
                else:
                    current_chunk = [sentence]
                    current_len = sent_len
            else:
                current_chunk.append(sentence)
                current_len += sent_len + 1
                
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        return chunks


## 3. Extraindo o Texto do PDF de Amostra
O código buscará qualquer PDF que veio junto do repositório, dentro da pasta `observatorio-ia/data/amostra`.

In [ ]:
import pdfplumber
import glob

# Busca os arquivos PDF dentro do projeto baixado
caminho_pasta_pdfs = '/content/observatorio-ia/data/amostra'
pdfs_encontrados = glob.glob(f"{caminho_pasta_pdfs}/**/*.pdf", recursive=True)

if not pdfs_encontrados:
    print(f"\nATENÇÃO: Nenhum PDF encontrado na pasta: {caminho_pasta_pdfs}")
else:
    print(f"\nForam encontrados {len(pdfs_encontrados)} PDFs. Usando o primeiro como exemplo...")
    pdf_filename = pdfs_encontrados[0]
    print(f"Arquivo alvo: '{pdf_filename}'\n")
    
    texto_bruto = ""
    with pdfplumber.open(pdf_filename) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                texto_bruto += t + "\n"
                
    texto_bruto = texto_bruto.replace('\x00', '').strip()

    print("--- TEXTO EXTRAÍDO (BRUTO) ---")
    print(texto_bruto[:1500])
    if len(texto_bruto) > 1500:
        print("...\n(texto truncado para exibição. Total de caracteres:", len(texto_bruto), ")")

## 4. O Processo de Limpeza
O texto extraído passa pela etapa de `clean_text` para retirar possíveis menus do site original, quebras de linhas erradas ou pontuações de PDF residuais.

In [ ]:
chunker = SemanticChunker()
texto_limpo = chunker.clean_text(texto_bruto)

print("--- TEXTO LIMPO ---")
print(texto_limpo[:1500])
if len(texto_limpo) > 1500:
    print("...\n(texto truncado para exibição. Total de caracteres:", len(texto_limpo), ")")

## 5. Divisão em Chunks (Semantic Segmentation)
Finalmente, usamos spaCy para entender a gramática da frase, retirar Stop Words, padronizar tudo e quebrar em blocos (chunks) menores que preservam a semântica de forma ideal para o modelo de RAG ou para Classificação.

In [ ]:
chunks = chunker.split_into_semantic_chunks(texto_bruto)

print("--- CHUNKS GERADOS ---")
print(f"Total de Chunks gerados: {len(chunks)}\n")

for idx, c in enumerate(chunks):
    if idx >= 10:
        print("... E assim por diante.")
        break
    print(f"CHUNK {idx+1}:")
    print(c)
    print("-" * 50)

## 6. Métricas de Qualidade e Extração
Para comprovar a eficiência do processo, o código abaixo calcula a quantidade de ruído (sujeira) retirado do PDF original e apresenta a distribuição dos blocos para garantir que atendem aos requisitos do modelo de IA.

In [ ]:
raw_len = len(texto_bruto)
clean_len = len(texto_limpo)
ruido_removido = raw_len - clean_len
reducao_percentual = (ruido_removido / raw_len) * 100 if raw_len > 0 else 0

total_chunks = len(chunks)
tamanhos_chunks = [len(c) for c in chunks]
media_tamanho_chunk = sum(tamanhos_chunks) / total_chunks if total_chunks > 0 else 0
chunk_max = max(tamanhos_chunks) if total_chunks > 0 else 0

palavras_bruto = len(texto_bruto.split())
palavras_limpo = len(texto_limpo.split())
palavras_chunks = sum(len(c.split()) for c in chunks)

print("\n📊 MÉTRICAS DE QUALIDADE DA EXTRAÇÃO E NLP")
print("="*60)
print(f"1. Tamanho do Texto Original (Bruto): {raw_len} caracteres ({palavras_bruto} palavras)")
print(f"2. Tamanho após Limpeza: {clean_len} caracteres ({palavras_limpo} palavras)")
print(f"3. Ruído e Formatação Removidos: {ruido_removido} caracteres")
print(f"   -> Taxa de Redução de Ruído: {reducao_percentual:.2f}% do PDF original era sujeira/layout de web")
print("-" * 60)
print(f"4. Total de Chunks Semânticos Gerados: {total_chunks}")
print(f"5. Média de Caracteres por Chunk: {media_tamanho_chunk:.1f} (Limite configurado: {chunker.chunk_size})")
print(f"6. Maior Chunk Gerado: {chunk_max} caracteres")
print(f"7. Densidade de Informação: {palavras_chunks} palavras condensadas e vetorizadas (sem stop-words)")
print("="*60)
print("✅ Conclusão: O pipeline limpa o ruído do PDF com eficiência e condensa")
print("a informação útil em blocos perfeitos para a IA (FAISS/RAG).\n")


## 7. Análise em Lote e Visualização Gráfica
Agora vamos processar **todos os PDFs da amostra** de uma só vez, contar quantos Chunks semânticos cada um gerou, e visualizar isso em um gráfico. Isso demonstra como o robô analisa grandes volumes de documentos.

In [ ]:
import matplotlib.pyplot as plt
import os

nomes_pdfs = []
quantidades_chunks = []

print(f"Processando {len(pdfs_encontrados)} arquivos em lote...")
for pdf_path in pdfs_encontrados:
    nome_arquivo = os.path.basename(pdf_path)
    
    texto_temp = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                texto_temp += t + "\n"
                
    texto_temp = texto_temp.replace('\x00', '').strip()
    chunks_temp = chunker.split_into_semantic_chunks(texto_temp)
    
    nomes_pdfs.append(nome_arquivo[:15] + ("..." if len(nome_arquivo)>15 else ""))
    quantidades_chunks.append(len(chunks_temp))

# Criando o Gráfico
plt.figure(figsize=(10, 6))
bars = plt.bar(nomes_pdfs, quantidades_chunks, color='#2c3e50', edgecolor='#34495e')

# Adicionando os números em cima de cada barra
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, int(yval), ha='center', va='bottom', fontweight='bold')

plt.title('Quantidade de Chunks Semânticos Gerados por Arquivo PDF', fontsize=14, fontweight='bold')
plt.xlabel('Nome do Arquivo PDF', fontsize=12)
plt.ylabel('Número de Chunks', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("\n✅ Análise concluída! Os documentos maiores naturalmente geram mais blocos (chunks),")
print("garantindo que o contexto seja quebrado em partes ideais para alimentar a IA.")
